# 01 — Data availability check: Gaza province, Mozambique

Goal: before building the full pipeline, confirm that SMAP soil moisture, CHIRPS precipitation,
and Sentinel-1 backscatter actually have usable coverage over Gaza for a period that includes a
known drought (e.g. 2023-2024), and that a drought signal is visible in the raw series.

Requires: `pip install earthengine-api geemap pandas matplotlib` and a prior
`earthengine authenticate` (opens a browser OAuth flow) plus a Google Cloud project with the
Earth Engine API enabled.

In [ ]:
import sys
sys.path.append("../src")

import ee
import matplotlib.pyplot as plt

import gee_extract as gx

ee.Initialize(project="geoprocessamento-426809")

In [ ]:
START = "2023-01-01"
END = "2024-12-31"
AOI = gx.GAZA_AOI

## SMAP surface soil moisture

In [ ]:
smap_df = gx.region_time_series(
    gx.COLLECTIONS["smap_soil_moisture"], "ssm", AOI, START, END, scale=10000
)
print(smap_df.shape)
smap_df.head()

## CHIRPS daily precipitation

In [ ]:
chirps_df = gx.region_time_series(
    gx.COLLECTIONS["chirps_precip"], "precipitation", AOI, START, END, scale=5000
)
print(chirps_df.shape)
chirps_df.head()

## Sentinel-2 NDVI (10 m native, reduced at 100 m for a province-sized AOI)

In [ ]:
ndvi_df = gx.sentinel2_ndvi_time_series(AOI, START, END, scale=100, max_cloud_pct=40)
print(ndvi_df.shape)
ndvi_df.head()

## Sentinel-1 VV backscatter

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

axes[0].plot(smap_df["date"], smap_df["ssm"], marker=".")
axes[0].set_ylabel("SMAP ssm")

axes[1].plot(chirps_df["date"], chirps_df["precipitation"], color="tab:blue")
axes[1].set_ylabel("CHIRPS mm/day")

axes[2].plot(ndvi_df["date"], ndvi_df["NDVI"], marker=".", color="tab:green")
axes[2].set_ylabel("S2 NDVI")

axes[3].plot(s1_df["date"], s1_df["VV"], marker=".", color="tab:orange")
axes[3].set_ylabel("S1 VV (dB)")
axes[3].set_xlabel("date")

fig.suptitle("Gaza, Mozambique — data availability check (2023-2024)")
fig.tight_layout()
plt.show()

## Checklist before moving to the full pipeline

- [ ] SMAP series has no large gaps (>2-3 weeks) over the study period
- [ ] CHIRPS shows a clear wet/dry seasonal cycle
- [ ] Sentinel-2 NDVI has enough cloud-free observations (check `ndvi_df.shape`) — southern
      Mozambique's wet season (Nov-Mar) will have more cloud gaps than the dry season
- [ ] Sentinel-1 has enough revisits (check `s1_df.shape`) — revisit is ~6-12 days depending on
      period/orbit availability over southern Africa
- [ ] A visible dip in SMAP/CHIRPS/NDVI aligns with the known 2023-2024 drought period — if not,
      double check the AOI and date range before building the full feature pipeline

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(smap_df["date"], smap_df["ssm"], marker=".")
axes[0].set_ylabel("SMAP ssm")

axes[1].plot(chirps_df["date"], chirps_df["precipitation"], color="tab:blue")
axes[1].set_ylabel("CHIRPS mm/day")

axes[2].plot(s1_df["date"], s1_df["VV"], marker=".", color="tab:orange")
axes[2].set_ylabel("S1 VV (dB)")
axes[2].set_xlabel("date")

fig.suptitle("Gaza, Mozambique — data availability check (2023-2024)")
fig.tight_layout()
plt.show()

## Checklist before moving to the full pipeline

- [ ] SMAP series has no large gaps (>2-3 weeks) over the study period
- [ ] CHIRPS shows a clear wet/dry seasonal cycle
- [ ] Sentinel-1 has enough revisits (check `s1_df.shape`) — revisit is ~6-12 days depending on
      period/orbit availability over southern Africa
- [ ] A visible dip in SMAP/CHIRPS aligns with the known 2023-2024 drought period — if not,
      double check the AOI and date range before building the full feature pipeline